In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Lecture du fichier Indicateur_emploi_agriculture
df_emploi_agriculture = pd.read_csv('/content/drive/MyDrive/Colab_Projet_Poule/Indicateur_emploi_agriculture.csv',decimal=".", index_col=0)
df_emploi_agriculture.head()

,Domaine,Code zone (FAO),Zone,Code indicateur,Indicateur,Code sexe,Sexe,Code année,Année,Code Élément,Élément,Code source,Source,Unité,Valeur,Symbole,Description du Symbole,Note
Code Domaine,,,,,,,,,,,,,,,,,,
OEA,Indicateurs de l’emploi: Agriculture et systèm...,2,Afghanistan,21160,Emploi total dans les systèmes agroalimentaire...,1,Total,2017,2017,6199,Valeur,3044,FAO Model,1000 No,4641.38,E,Valeur estimée,Modelled using ISIC shares
OEA,Indicateurs de l’emploi: Agriculture et systèm...,2,Afghanistan,21160,Emploi total dans les systèmes agroalimentaire...,1,Total,2018,2018,6199,Valeur,3044,FAO Model,1000 No,4691.69,E,Valeur estimée,Modelled FAO
OEA,Indicateurs de l’emploi: Agriculture et systèm...,2,Afghanistan,21160,Emploi total dans les systèmes agroalimentaire...,1,Total,2019,2019,6199,Valeur,3044,FAO Model,1000 No,4820.24,E,Valeur estimée,Modelled FAO
OEA,Indicateurs de l’emploi: Agriculture et systèm...,2,Afghanistan,21160,Emploi total dans les systèmes agroalimentaire...,1,Total,2020,2020,6199,Valeur,3044,FAO Model,1000 No,4821.39,E,Valeur estimée,Modelled FAO
OEA,Indicateurs de l’emploi: Agriculture et systèm...,2,Afghanistan,21160,Emploi total dans les systèmes agroalimentaire...,1,Total,2021,2021,6199,Valeur,3044,FAO Model,1000 No,4827.20,E,Valeur estimée,Modelled FAO


In [ ]:
#Verification de type des variables
df_emploi_agriculture.dtypes

,0
Domaine,object
Code zone (FAO),int64
Zone,object
Code indicateur,int64
Indicateur,object
Code sexe,int64
Sexe,object
Code année,int64
Année,int64
Code Élément,int64


In [ ]:
# Filtrer pour l'Indicateur Emploi dans la production végétale et animale, la chasse et les services connexes

df_prod_animale = df_emploi_agriculture.loc[df_emploi_agriculture['Code indicateur'] == 21111, :]
variable_prod_animale = ['Code zone (FAO)','Zone', 'Année', 'Valeur']
df_prod_animale = df_prod_animale[variable_prod_animale]
df_prod_animale.shape

(650, 4)

In [ ]:
# Groupe de zone avec leurs indicateurs Emploi dans la production végétale et animale, la chasse et les services connexes
data_prod_animale = df_prod_animale.pivot_table(index= ['Code zone (FAO)','Zone'], columns= ['Année'], values = 'Valeur')
data_prod_animale.reset_index(inplace=True)
display(data_prod_animale)

Année,Code zone (FAO),Zone,2017,2018,2019,2020,2021,2022,2023,2024
0,1,Arménie,379.75,387.06,348.58,361.06,330.63,340.11,339.74,NaN
1,2,Afghanistan,2737.94,NaN,NaN,NaN,1185.84,NaN,NaN,NaN
2,3,Albanie,450.26,451.79,455.85,441.43,416.74,437.61,421.64,NaN
3,7,Angola,NaN,NaN,5238.07,NaN,6130.84,5998.46,NaN,NaN
4,9,Argentine,56.52,52.74,58.83,44.89,57.78,69.37,68.22,NaN
...,...,...,...,...,...,...,...,...,...,...
137,255,Belgique,51.36,45.32,43.31,42.82,44.28,42.99,52.69,NaN
138,256,Luxembourg,3.15,2.64,1.84,2.08,3.07,3.09,2.63,NaN
139,272,Serbie,465.16,436.16,433.37,NaN,NaN,NaN,NaN,NaN
140,276,Soudan,NaN,NaN,NaN,NaN,NaN,3100.30,NaN,NaN


In [ ]:
#vérification des valeurs manquantes
print(data_prod_animale.isnull().sum())

Année
Code zone (FAO)      0
Zone                 0
2017                49
2018                54
2019                41
2020                55
2021                51
2022                46
2023                61
2024               129
dtype: int64


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks')

In [ ]:
import importlib
import Parcourir

importlib.reload(Parcourir)

<module 'Parcourir' from '/content/drive/MyDrive/Colab Notebooks/Parcourir.py'>

In [ ]:
import re

#Copie du dataframe data_prod_animale
prod_animale = data_prod_animale.copy()

#détecter les colonnes année (ex : 2000 [YR2000])
year_cols = [c for c in prod_animale.columns if re.search(r'\b\d{4}\b', str(c))]
pd.set_option('future.no_silent_downcasting', True)
#normaliser placeholders en NaN (ex: '..', '', 'NA')
prod_animale[year_cols] = prod_animale[year_cols].replace(['..', '', 'NA', 'na', 'N/A'], np.nan)

In [ ]:
 #remplissage des valeurs NaN en 2024 par la valeur la plus récente
prod_animale[2024] = prod_animale[year_cols].apply(Parcourir.last_valid_value_from_right, axis=1)

In [ ]:
#Affichage du dataframe prod_animale avec 3 variables seulement
var_prod_animale = ['Code zone (FAO)','Zone',2024]
prod_animale = prod_animale[var_prod_animale]
prod_animale.head()

Année,Code zone (FAO),Zone,2024
0,1,Arménie,339.74
1,2,Afghanistan,1185.84
2,3,Albanie,421.64
3,7,Angola,5998.46
4,9,Argentine,68.22


In [ ]:
# Renommage des colonnes de la dataframe prod_animale
prod_animale = prod_animale.rename(columns={'Code zone (FAO)': 'Code zone', 2024:'Emploi dans la production_vegetale_animale (1000No)' })
display(prod_animale)

Année,Code zone,Zone,Emploi dans la production_vegetale_animale (1000No)
0,1,Arménie,339.74
1,2,Afghanistan,1185.84
2,3,Albanie,421.64
3,7,Angola,5998.46
4,9,Argentine,68.22
...,...,...,...
137,255,Belgique,52.69
138,256,Luxembourg,2.63
139,272,Serbie,433.37
140,276,Soudan,3100.30


In [ ]:
#vérification des valeurs manquantes
print(prod_animale.isnull().sum())

Année
Code zone                                              0
Zone                                                   0
Emploi dans la production_vegetale_animale (1000No)    0
dtype: int64


In [ ]:
#Vérification des Doublons
prod_animale.loc[prod_animale[['Code zone','Zone','Emploi dans la production_vegetale_animale (1000No)']].duplicated(keep=False),:]

Année,Code zone,Zone,Emploi dans la production_vegetale_animale (1000No)


In [ ]:
#Verification s'il y a des outliers
prod_animale.describe()

Année,Code zone,Emploi dans la production_vegetale_animale (1000No)
count,142.000000,142.000000
mean,129.725352,3913.143380
std,73.901904,17624.319492
min,1.000000,0.030000
25%,69.250000,54.192500
50%,128.000000,371.595000
75%,192.250000,2204.705000
max,299.000000,201764.690000


In [ ]:
# Tri des valeurs par ordre décroissant
prod_animale_ordre = prod_animale.sort_values('Emploi dans la production_vegetale_animale (1000No)', ascending= False)
display(prod_animale_ordre)

Année,Code zone,Zone,Emploi dans la production_vegetale_animale (1000No)
50,100,Inde,201764.69
51,101,Indonésie,37364.68
9,16,Bangladesh,27670.18
89,159,Nigéria,24203.79
134,238,Éthiopie,23506.69
...,...,...,...
23,47,Îles Cook,0.17
126,227,Tuvalu,0.12
90,160,Nioué,0.07
79,142,Montserrat,0.04


In [ ]:
#préparation de l'export du dataframe en fichier CSV

prod_animale.to_csv('/content/drive/MyDrive/Colab Notebooks/Emlpoi_prod_animale.csv', index=False)